# Evaluación Sumativa 1 — Framingham Heart Study

Grupo 3 — MCDI501: Estadística Computacional para la Toma de Decisiones

Este cuaderno acompaña al informe técnico de la Sumativa 1 y contiene el código completo y reproducible.
Sigue el mismo orden del informe: (1) preparación y calidad de los datos, (2) análisis exploratorio
(descriptivo y bivariado), (3) estimación de parámetros y (4) pruebas de hipótesis.

Retoma y profundiza el trabajo de la Formativa 1 (`informes/formativa-1-eda-inferencia/`): el conjunto de datos,
la variable objetivo y los resultados descriptivos son los mismos; aquí se agregan las decisiones de limpieza,
el análisis bivariado con variables derivadas, una estimación adicional, la verificación de supuestos y una
tercera prueba de hipótesis. Semilla fija: `42`.

**Problema y objetivo de esta fase.** Se estudian factores asociados a la enfermedad coronaria (CHD)
a diez años, con la anticipación del riesgo en atención primaria como contexto de aplicación propuesto.
Esta fase caracteriza los datos de Framingham, estima parámetros y contrasta asociaciones preliminares;
todavía no valida un predictor ni establece relaciones causales.

## 0. Configuración y carga de datos
Ejecutar desde la raíz del repositorio o desde la carpeta de este notebook, con las dependencias de
`requirements.txt`. Se reutiliza `data/raw/framingham.csv` si existe; solo la primera carga requiere
internet. Se registra la huella SHA-256 y las versiones para identificar datos y entorno.
Las figuras se generan en `redaccion/sumativa-1-analisis-exploratorio-inferencial/figuras/`.
Para Canvas se entrega una copia ejecutada con salidas; el repositorio puede conservarla sin salidas por `nbstripout`.

In [ ]:
from pathlib import Path
import hashlib, platform, importlib.metadata
from IPython.display import display
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import FuncFormatter
from scipy import stats

In [ ]:
# Localizar el repositorio sin depender de rutas relativas ../../
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "requirements.txt").exists() and (p / "redaccion").is_dir()), None)
if ROOT is None:
    raise FileNotFoundError("Abra el notebook dentro del repositorio del proyecto.")
CSV = ROOT / "data/raw/framingham.csv"
if not CSV.exists():
    import os, shutil
    os.environ["TQDM_DISABLE"] = "1"
    import warnings, io
    from contextlib import redirect_stdout
    with warnings.catch_warnings(), redirect_stdout(io.StringIO()):
        warnings.filterwarnings("ignore", message="IProgress not found.*")
        import kagglehub
        downloaded = kagglehub.dataset_download("dileep070/heart-disease-prediction-using-logistic-regression")
    CSV.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(Path(downloaded) / "framingham.csv", CSV)
print("Dataset disponible: data/raw/framingham.csv")
print("SHA-256:", hashlib.sha256(CSV.read_bytes()).hexdigest())
display(pd.DataFrame({"Componente": ["Python", "numpy", "pandas", "scipy", "matplotlib", "kagglehub"],
    "Versión": [platform.python_version(), *[importlib.metadata.version(p)
                 for p in ["numpy", "pandas", "scipy", "matplotlib", "kagglehub"]]]}))

In [ ]:
SEED = 42
np.random.seed(SEED)
TARGET = "TenYearCHD"

df = pd.read_csv(CSV)
n = len(df)
ORIGINALES = df.columns.tolist()
ALPHA = 0.05
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Las figuras se guardan directo en redaccion/, de donde las toma informe.tex al compilar
FIGURAS = ROOT / "redaccion/sumativa-1-analisis-exploratorio-inferencial/figuras"
FIGURAS.mkdir(parents=True, exist_ok=True)

# Estilo común de las figuras (misma paleta UNAB de la Formativa 1)
NAVY, RED, GREY, BG = "#0d2741", "#ae0000", "#8c8578", "#f6f1ea"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 200,
    "font.size": 10, "axes.facecolor": BG,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.titlesize": 12, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.grid": True, "grid.color": "#ddd5c6", "axes.axisbelow": True,
})
# Nombres en español para tablas y figuras
ETIQ = {
    "male": "Sexo masculino", "age": "Edad", "education": "Nivel educativo",
    "currentSmoker": "Fumador actual", "cigsPerDay": "Cigarrillos por día",
    "BPMeds": "Medicación antihipertensiva", "prevalentStroke": "ACV previo",
    "prevalentHyp": "Hipertensión prevalente", "diabetes": "Diabetes",
    "totChol": "Colesterol total", "sysBP": "Presión sistólica", "diaBP": "Presión diastólica",
    "BMI": "IMC", "heartRate": "Frecuencia cardíaca", "glucose": "Glucosa",
}

def fmt_es(x, dec=1):
    """Número con formato español: coma decimal y punto de miles."""
    return f"{x:,.{dec}f}".replace(",", "X").replace(".", ",").replace("X", ".")

def eje_es(ax, eje="y", dec=0):
    """Aplica el formato español a los números del eje indicado."""
    f = FuncFormatter(lambda v, _: fmt_es(v, dec))
    (ax.yaxis if eje == "y" else ax.xaxis).set_major_formatter(f)

print("Filas y columnas:", df.shape, "| Semilla:", SEED, "| Objetivo:", TARGET)
df.head()

## 1. Preparación y calidad de los datos
Antes de cualquier análisis se revisan la estructura, los tipos de variable, los datos faltantes, los
duplicados, las inconsistencias entre variables relacionadas y la plausibilidad de los valores extremos.

In [ ]:
# Estructura y tipos de variable
tipos = pd.DataFrame({
    "Tipo": ["Binaria", "Cuasi-continua", "Ordinal (1-4)", "Binaria", "Conteo", "Binaria", "Binaria",
             "Binaria", "Binaria", "Continua", "Continua", "Continua", "Continua", "Continua",
             "Continua", "Binaria (objetivo)"],
    "dtype": df.dtypes.astype(str).values,
    "Valores no nulos": df.notna().sum().values,
}, index=df.columns)
tipos

### Diccionario de variables
| Variable | Tipo / unidad o código |
|---|---|
| `age` | Edad en años |
| `male` | Código de sexo: 0 mujer, 1 hombre |
| `education` | Nivel ordinal codificado de 1 a 4; no se asumen distancias iguales |
| `currentSmoker`, `BPMeds`, `prevalentStroke`, `prevalentHyp`, `diabetes` | Indicadores: 0 no, 1 sí |
| `cigsPerDay` | Cigarrillos por día, conteo |
| `totChol`, `glucose` | mg/dL |
| `sysBP`, `diaBP` | mmHg |
| `BMI` | kg/m² |
| `heartRate` | Latidos por minuto |
| `TenYearCHD` | 0 sin CHD, 1 con CHD en diez años |
Los nombres y códigos corresponden al archivo utilizado; no se infieren detalles del diseño muestral no documentados.

**Lectura.** El conjunto tiene 4.238 filas y 16 variables, todas numéricas en el archivo: el objetivo
binario `TenYearCHD`, seis indicadores binarios, una variable ordinal (`education`, niveles 1 a 4), un conteo
(`cigsPerDay`) y siete variables numéricas (seis continuas y la edad, cuasi-continua). Las variables con
faltantes se leen como `float64`; las binarias y la ordinal se analizan como categóricas.

In [ ]:
# Datos faltantes y su relación con la variable objetivo
falt = pd.DataFrame({
    "n faltantes": df.isna().sum(),
    "% faltante": (df.isna().mean() * 100).round(2),
    "% CHD con faltante": [df.loc[df[c].isna(), TARGET].mean() * 100 if df[c].isna().any() else np.nan for c in df],
    "% CHD sin faltante": [df.loc[df[c].notna(), TARGET].mean() * 100 for c in df],
}).round(1)
falt = falt[falt["n faltantes"] > 0].sort_values("n faltantes", ascending=False)
display(falt)

completos = df.dropna()
print(f"Filas con al menos un faltante: {n - len(completos):,} ({(n - len(completos)) / n * 100:.1f}%)")
print(f"Casos completos               : {len(completos):,} ({len(completos) / n * 100:.1f}%)\n")

# ¿El faltante de glucosa depende del objetivo? (indicio sobre el mecanismo de pérdida)
falta_glu = df["glucose"].isna()
chi2_g, p_g, _, _ = stats.chi2_contingency(pd.crosstab(falta_glu, df[TARGET]))
print(f"Glucosa faltante vs. CHD: chi2 = {chi2_g:.2f}, p = {p_g:.3f}")
print(f"Diabéticos entre quienes no tienen glucosa: {df.loc[falta_glu, 'diabetes'].mean() * 100:.1f}% "
      f"(vs. {df.loc[~falta_glu, 'diabetes'].mean() * 100:.1f}% con dato)")
print("cigsPerDay faltante: fumadores actuales =", int(df.loc[df["cigsPerDay"].isna(), "currentSmoker"].sum()),
      "de", int(df["cigsPerDay"].isna().sum()))

**Lectura.** Hay faltantes en siete variables. La glucosa concentra la mayor parte (388 registros, 9,2%),
seguida del nivel educativo (105, 2,5%); el resto no supera el 1,3%. En total, 582 filas (13,7%) tienen al menos
un faltante, de modo que eliminar filas incompletas dejaría 3.656 casos (86,3%). El faltante de glucosa no se
asocia con el objetivo (p = 0,209): la tasa de CHD es parecida con y sin dato, sin evidencia suficiente para rechazar la independencia. Esto no demuestra ausencia aleatoria
ni descarta sesgo; se usan casos disponibles como decisión exploratoria y se reconoce esa limitación. Los 29 faltantes de `cigsPerDay` corresponden todos a fumadores
actuales, información útil para imputarlos más adelante. Las tasas de CHD de los grupos con muy pocos
faltantes (IMC, frecuencia cardíaca) no son interpretables por su tamaño.

In [ ]:
# Duplicados, inconsistencias y plausibilidad de valores extremos
print("Filas duplicadas:", df.duplicated().sum())
print("diaBP >= sysBP:", (df["diaBP"] >= df["sysBP"]).sum())
print("Fumador con 0 cig/día:", ((df["cigsPerDay"] == 0) & (df["currentSmoker"] == 1)).sum())
print("No fumador con cig/día > 0:", ((df["cigsPerDay"] > 0) & (df["currentSmoker"] == 0)).sum())
print("Binarias fuera de {0,1}:", sum((~df[c].dropna().isin([0, 1])).sum() for c in
      ["male", "currentSmoker", "BPMeds", "prevalentStroke", "prevalentHyp", "diabetes", TARGET]))
print()

# Valores extremos según Tukey y revisión clínica
for c in ["totChol", "sysBP", "diaBP", "BMI", "glucose"]:
    q1, q3 = df[c].quantile([0.25, 0.75])
    k = ((df[c] < q1 - 1.5 * (q3 - q1)) | (df[c] > q3 + 1.5 * (q3 - q1))).sum()
    print(f"Tukey {c:8s}: {k:4d} ({k / df[c].notna().sum() * 100:.1f}%)   rango [{df[c].min():g}; {df[c].max():g}]")
g200 = df["glucose"] >= 200
print(f"\nGlucosa >= 200 mg/dL: {g200.sum()} registros, de ellos diabéticos: {int(df.loc[g200, 'diabetes'].sum())}")
print("Presión sistólica >= 200 mmHg:", (df["sysBP"] >= 200).sum(), "| colesterol >= 400 mg/dL:", (df["totChol"] >= 400).sum())

In [ ]:
# Validaciones de dominio y reporte inicial consolidado
reglas = {"education fuera de 1-4": (~df["education"].dropna().isin([1,2,3,4])).sum(),
          "cigsPerDay negativo": (df["cigsPerDay"] < 0).sum()}
for c in ["age", "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"]:
    reglas[f"{c} no positivo"] = (df[c] <= 0).sum()
display(pd.DataFrame(reglas.items(), columns=["Regla de consistencia", "n incumplimientos"]))
calidad = pd.DataFrame({"Variable": ORIGINALES,
    "n válidos": [df[c].notna().sum() for c in ORIGINALES],
    "n faltantes": [df[c].isna().sum() for c in ORIGINALES],
    "% faltantes": [df[c].isna().mean()*100 for c in ORIGINALES],
    "Tratamiento": ["Casos disponibles; sin imputación" if df[c].isna().any()
                    else "Conservar" for c in ORIGINALES]})
display(calidad.round(2))
print("Estas reglas detectan errores de dominio; no certifican plausibilidad clínica de cada registro.")

**Lectura y decisiones de tratamiento.** No hay filas duplicadas ni inconsistencias entre variables
relacionadas: ninguna presión diastólica iguala o supera a la sistólica, el hábito de fumar es coherente con los
cigarrillos por día y todas las binarias toman solo los valores 0 y 1. Los valores extremos según Tukey van de
1,3% (colesterol) a 4,9% (glucosa) y son clínicamente plausibles: los 34 registros con glucosa de 200 mg/dL o
más corresponden todos a personas diabéticas. Por eso se conservan, y para las variables asimétricas se usa la
mediana como referencia robusta. Las decisiones se resumen así:

#### Tabla 1 - Decisiones de limpieza y tratamiento de datos

| Situación | Decisión |
|---|---|
| Faltantes (7 variables; 13,7% de las filas) | Se trabaja con los casos disponibles de cada variable; la imputación se aborda en el modelamiento. |
| Faltantes de glucosa (9,2%) | No se detectó asociación con CHD (p = 0,21), sin identificar el mecanismo de ausencia; se usan casos disponibles. |
| `cigsPerDay` faltante (29, todos fumadores) | Se documenta; podrá imputarse con la mediana de los fumadores. |
| Valores extremos (Tukey) | Se conservan por ser plausibles (p. ej., glucosa ≥ 200 solo en diabéticos); se usa la mediana como referencia y los gráficos muestran los valores completos, sin modificar la cola. |
| Duplicados e inconsistencias | No se detectaron. |

## 2. Análisis exploratorio de datos

### 2.1 Estadística descriptiva
#### Tabla 2 - Medidas resumen de las variables numéricas

In [ ]:
cuant = ["age", "cigsPerDay", "totChol", "sysBP", "diaBP", "BMI", "heartRate", "glucose"]
desc = df[cuant].describe().T[["mean", "50%", "std", "min", "max", "25%", "75%"]]
desc["skew"] = df[cuant].skew()
desc["n"] = df[cuant].notna().sum()
desc.columns = ["Media", "Mediana", "Desv. est.", "Mínimo", "Máximo", "Q1", "Q3", "Asimetría", "n"]
desc["RIC"] = desc["Q3"] - desc["Q1"]
desc.round(2)

**Lectura.** La edad es la variable más simétrica (media 49,58 y mediana 49 años; asimetría 0,23). El
colesterol total, la presión sistólica y el IMC tienen asimetría moderada a la derecha (0,87; 1,15 y 0,98), con la
media por sobre la mediana, y la glucosa es la más sesgada (6,21), arrastrada por los valores altos de las
personas diabéticas. En `cigsPerDay` la mediana es 0 y la media 9: más de la mitad no fuma y la media la elevan
los fumadores intensos. Estas formas justifican usar la mediana como referencia en las variables asimétricas.

#### Tabla 3 - Variables binarias, ordinal y tasa de CHD

In [ ]:
bin_vars = ["male", "currentSmoker", "BPMeds", "prevalentStroke", "prevalentHyp", "diabetes"]
tabla_bin = pd.DataFrame({
    "n con atributo": [(df[c] == 1).sum() for c in bin_vars],
    "n sin atributo": [(df[c] == 0).sum() for c in bin_vars],
    "n válidos": df[bin_vars].notna().sum(), "n faltantes": df[bin_vars].isna().sum(),
    "% con atributo": df[bin_vars].mean() * 100,
    "% CHD con atributo": [df.loc[df[c] == 1, TARGET].mean() * 100 for c in bin_vars],
    "% CHD sin atributo": [df.loc[df[c] == 0, TARGET].mean() * 100 for c in bin_vars],
}).round(1)
tabla_bin.index = [ETIQ[c] for c in bin_vars]
display(tabla_bin)

filas_cat = []
for c in [*bin_vars, "education", TARGET]:
    validos = df[c].notna().sum()
    for categoria, conteo in df[c].value_counts().sort_index().items():
        filas_cat.append({"Variable": c, "Categoría": int(categoria), "n": conteo,
                          "% válido": conteo / validos * 100, "n válidos": validos,
                          "n faltantes": df[c].isna().sum()})
frecuencias_cat = pd.DataFrame(filas_cat)
display(frecuencias_cat.round(2))
edu = pd.DataFrame({"n": df["education"].value_counts().sort_index(),
    "% de participantes": df["education"].value_counts(normalize=True).sort_index() * 100,
    "% CHD": df.groupby("education")[TARGET].mean() * 100})
display(edu.round(1))

**Lectura.** El 42,9% son hombres, el 49,4% fuma actualmente y el 31,1% tiene hipertensión prevalente;
la medicación antihipertensiva (3,0%), la diabetes (2,6%) y el ACV previo (0,6%) son poco frecuentes. La tasa de
CHD es mayor en quienes tienen cada antecedente, mientras que fumar actualmente casi no la cambia (15,9% frente a
14,5%). El nivel educativo se concentra en el nivel 1 (41,6%), que es también el de mayor tasa de CHD (18,8%).
Son comparaciones descriptivas, sin ajustar por otras variables; los grupos con ACV previo o diabetes son
pequeños y sus tasas, menos precisas.

#### Figura adicional - Frecuencias de las variables categóricas

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6.5))
for ax, c in zip(axes.flat, [*bin_vars, "education", TARGET]):
    sub = frecuencias_cat[frecuencias_cat["Variable"] == c]
    labels = (["Mujer", "Hombre"] if c == "male" else
              ["No", "Sí"] if c != "education" else [str(x) for x in sub["Categoría"]])
    ax.bar(labels, sub["% válido"], color=NAVY)
    for i, (_, row) in enumerate(sub.iterrows()):
        ax.text(i, row["% válido"] + 2, f"{fmt_es(row['% válido'])}%\nn={fmt_es(row['n'],0)}",
                ha="center", fontsize=8)
    ax.set_ylim(0, 115)
    ax.set_title(ETIQ.get(c, "CHD a 10 años"), fontsize=10)
    ax.set_ylabel("% de casos válidos")
    ax.grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(FIGURAS / "fig_categoricas.png", bbox_inches="tight")
plt.show()

# Versión compacta para el informe: atributos presentes y niveles educativos.
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
datos = tabla_bin["% con atributo"]
ax[0].barh(datos.index, datos.values, color=NAVY)
ax[0].invert_yaxis()
for i, v in enumerate(datos):
    ax[0].text(v+1, i, f"{fmt_es(v)}%", va="center", fontsize=9)
ax[0].set_xlim(0, 72)
ax[0].set_xlabel("% con atributo (casos válidos)")
ax[0].set_title("(a) Indicadores binarios")
ax[0].grid(axis="y", visible=False)
ax[1].bar([str(int(x)) for x in edu.index], edu["% de participantes"], color=NAVY)
for i, (_, row) in enumerate(edu.iterrows()):
    ax[1].text(i, row["% de participantes"]+1, f"{fmt_es(row['% de participantes'])}%\nn={fmt_es(row['n'],0)}", ha="center", fontsize=9)
ax[1].set_ylim(0, 53)
ax[1].set_xlabel("Nivel educativo (códigos ordinales)")
ax[1].set_ylabel("% de casos válidos")
ax[1].set_title("(b) Nivel educativo")
ax[1].grid(axis="x", visible=False)
fig.tight_layout()
fig.savefig(FIGURAS / "fig_categoricas_informe.png", bbox_inches="tight")
plt.show()


#### Figura 1 - Variable objetivo y datos faltantes

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

# (a) distribución de la variable objetivo
vc = df[TARGET].value_counts().sort_index()
ax[0].barh(["Sin CHD (0)", "Con CHD (1)"], vc.values, color=[NAVY, RED], height=0.5)
ax[0].invert_yaxis()
for i, v in enumerate(vc.values):
    ax[0].text(v + 50, i, f"{fmt_es(v, 0)}  ({fmt_es(v / n * 100)}%)", va="center", fontsize=9)
ax[0].set_xlim(0, vc.max() * 1.3)
ax[0].set_xlabel("Número de participantes")
ax[0].set_title("(a) Distribución de la variable objetivo")
ax[0].grid(axis="y", visible=False)
eje_es(ax[0], "x")

# (b) porcentaje de faltantes por variable
miss = (df.isna().mean() * 100).sort_values(ascending=False)
miss = miss[miss > 0]
y = np.arange(len(miss))
ax[1].hlines(y, 0, miss.values, color="#bdb3a3", lw=2.5)
ax[1].scatter(miss.values, y, s=90, color=RED, edgecolor="#333333", linewidth=0.6, zorder=3)
for i, v in enumerate(miss.values):
    ax[1].text(v + 0.2, i, f"{fmt_es(v)}%", va="center", fontsize=9)
ax[1].set_yticks(y)
ax[1].set_yticklabels([ETIQ[c] for c in miss.index], fontsize=9)
ax[1].invert_yaxis()
ax[1].set_xlim(0, miss.max() * 1.25)
ax[1].set_xlabel("% de datos faltantes")
ax[1].set_title("(b) Datos faltantes por variable")
ax[1].grid(axis="y", visible=False)
eje_es(ax[1], "x")

plt.tight_layout()
plt.savefig(FIGURAS / "fig_panorama.png", bbox_inches="tight")
plt.show()

**Lectura.** El objetivo está desbalanceado: 644 de 4.238 participantes (15,2%) desarrollan CHD, unas
5,6 personas sanas por cada caso. Predecir siempre "sin CHD" acertaría el 84,8% de las veces, por lo que los
modelos de las fases siguientes deberán evaluarse con métricas sensibles a la clase minoritaria.

#### Figura 2 - Distribución de tres variables numéricas
El colesterol total se recorta al percentil 99 solo para visualizarlo; los cálculos usan todos los datos.

In [ ]:
chol = df["totChol"].dropna()
sbp = df["sysBP"]
paneles = [
    (df["age"], "Edad", "años", "Media", df["age"].mean()),
    (chol, "Colesterol total", "mg/dL", "Mediana", chol.median()),
    (sbp, "Presión sistólica", "mmHg", "Mediana", sbp.median()),
]

fig, axes = plt.subplots(1, 3, figsize=(13, 3.9))
for ax, (x, titulo, xlab, nombre, ref) in zip(axes, paneles):
    ax.hist(x, bins=(np.arange(31.5, 71.5, 1) if titulo == "Edad" else 40), histtype="stepfilled",
            color=NAVY, alpha=0.3, edgecolor=NAVY, linewidth=1.2, label="Participantes")
    ax.axvline(ref, color=RED, ls="--", lw=1.8, label=f"{nombre} = {fmt_es(ref, 0)} {xlab}")
    ax.set_title(titulo)
    ax.set_xlabel(xlab)
    ax.grid(axis="x", visible=False)
    ax.set_ylim(0, ax.get_ylim()[1] * 1.25)
    ax.legend(frameon=False, loc="upper right", fontsize=8.5)
    eje_es(ax, "y")
axes[0].set_ylabel("Frecuencia (n.º de participantes)")

plt.tight_layout()
plt.savefig(FIGURAS / "fig_distribuciones.png", bbox_inches="tight")
plt.show()

**Lectura.** La edad se reparte de forma aproximadamente simétrica en torno a 50 años; el colesterol total y
la presión sistólica son asimétricos a la derecha, con la mediana (234 mg/dL y 128 mmHg) bajo la media y una
cola de valores altos.

### 2.2 Análisis bivariado
#### Figura 3 - Matriz de correlaciones (triángulo inferior)

#### Distribuciones completas de las ocho variables numéricas

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 6.5))
for ax, c in zip(axes.flat, cuant):
    x = df[c].dropna()
    ax.hist(x, bins=35, color=NAVY, alpha=0.7)
    ax.axvline(x.median(), color=RED, ls="--", label="Mediana")
    ax.set_title(ETIQ[c], fontsize=10)
    ax.set_ylabel("Frecuencia")
    ax.legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIGURAS / "fig_numericas_completo.png", bbox_inches="tight")
plt.show()
print("Se muestran todos los valores observados; las colas largas pueden concentrar visualmente el resto.")

In [ ]:
corr = df[[c for c in ORIGINALES if c != TARGET]].corr()
k = len(corr)
mask = np.triu(np.ones((k, k), dtype=bool), k=1)   # oculta el triángulo superior
cmap = LinearSegmentedColormap.from_list("div", [RED, "#ede4d6", NAVY])

fig, ax = plt.subplots(figsize=(9, 8))
ax.set_facecolor("white")
ax.grid(False)
im = ax.pcolormesh(np.ma.masked_array(corr.values, mask), cmap=cmap, vmin=-1, vmax=1,
                   edgecolors="white", linewidth=1.5)
ax.invert_yaxis()
for i in range(k):
    for j in range(i + 1):
        v = corr.values[i, j]
        ax.text(j + 0.5, i + 0.5, fmt_es(v, 2), ha="center", va="center", fontsize=7,
                color="white" if abs(v) > 0.5 else "#333333")
ax.set_xticks(np.arange(k) + 0.5)
ax.set_xticklabels([ETIQ[c] for c in corr.columns], rotation=90, fontsize=8)
ax.set_yticks(np.arange(k) + 0.5)
ax.set_yticklabels([ETIQ[c] for c in corr.columns], fontsize=8)
ax.set_title("Matriz de correlaciones entre variables")
cb = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cb.set_label("Coeficiente de correlación de Pearson (r)")

plt.tight_layout()
plt.savefig(FIGURAS / "fig_correlacion.png", bbox_inches="tight")
plt.show()

# Pares con mayor |r| fuera de la diagonal
pares = corr.where(np.tril(np.ones((k, k), dtype=bool), k=-1)).stack()
print("Pares con mayor |r|:")
display(pares.reindex(pares.abs().sort_values(ascending=False).index).head(5).round(3).to_frame("r"))

# Correlación de cada variable con el objetivo
print("Correlación con TenYearCHD:")
display(df[ORIGINALES].corr()[TARGET].drop(TARGET).sort_values().round(3).to_frame("r"))

**Lectura.** Las correlaciones más altas son presión sistólica con diastólica (r = 0,78), fumador actual con
cigarrillos por día (r = 0,77, en parte estructural), hipertensión prevalente con presión sistólica (r = 0,70) y
diabetes con glucosa (r = 0,62): anticipan colinealidad si se incluyen juntas en un modelo. Con el objetivo,
todas las correlaciones lineales son débiles; las mayores son la edad (r = 0,23), la presión sistólica (r = 0,22)
y la hipertensión prevalente (r = 0,18). Cuando una de las variables es binaria, el coeficiente de Pearson
equivale al punto biserial (binaria con numérica) o al coeficiente phi (binaria con binaria).

#### Variables categóricas derivadas
Para ver relaciones que la correlación lineal no capta, se construyen dos variables categóricas a partir de
las numéricas: el tramo etario (decenios) y la categoría de presión sistólica, con cortes cercanos a los
umbrales clínicos habituales (cortes descriptivos de 120, 130, 140 y 160 mmHg; no constituyen un diagnóstico clínico).

In [ ]:
df["tramo_edad"] = pd.cut(df["age"], bins=[30, 40, 50, 60, 71], right=False,
                         labels=["32-39", "40-49", "50-59", "60-70"])
df["cat_pas"] = pd.cut(df["sysBP"], bins=[0, 120, 130, 140, 160, 400], right=False,
                      labels=["<120", "120-129", "130-139", "140-159", "≥160"])
gm = df[TARGET].mean() * 100

tasa_edad = df.groupby("tramo_edad", observed=True)[TARGET].agg(["size", "mean"])
tasa_pas = df.groupby("cat_pas", observed=True)[TARGET].agg(["size", "mean"])
tasa_edad["mean"] *= 100
tasa_pas["mean"] *= 100
print(f"Tasa global de CHD: {gm:.1f}%\n")
display(tasa_edad.round(1))
display(tasa_pas.round(1))

#### Figura 4 - Tasa de CHD por tramo etario y por categoría de presión sistólica

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

def lollipop(a, tasa, titulo, xlab):
    xx = np.arange(len(tasa))
    a.axhline(gm, color=GREY, ls="--", lw=1.5, label=f"Media global = {fmt_es(gm)}%")
    a.vlines(xx, 0, tasa["mean"].values, color="#bdb3a3", lw=2.5)
    a.scatter(xx, tasa["mean"].values, s=95, color=RED, edgecolor="#333333", linewidth=0.6, zorder=3)
    for xi, yi in zip(xx, tasa["mean"].values):
        a.text(xi, yi + tasa["mean"].max() * 0.05, f"{fmt_es(yi)}%", ha="center", fontsize=9)
    a.set_xticks(xx)
    a.set_xticklabels([f"{c}\n(n={fmt_es(s, 0)})" for c, s in zip(tasa.index, tasa["size"])], fontsize=8.5)
    a.set_xlabel(xlab)
    a.set_ylabel("% con CHD a 10 años")
    a.set_ylim(0, tasa["mean"].max() * 1.25)
    a.set_title(titulo)
    a.grid(axis="x", visible=False)
    a.legend(frameon=False, loc="upper left", fontsize=8.5)
    eje_es(a, "y")

lollipop(ax[0], tasa_edad, "(a) Por tramo etario", "Edad (años)")
lollipop(ax[1], tasa_pas, "(b) Por presión sistólica", "Presión sistólica (mmHg)")
plt.tight_layout()
plt.savefig(FIGURAS / "fig_bivariado.png", bbox_inches="tight")
plt.show()

**Lectura.** La tasa de CHD crece de forma sostenida con la edad, del 4,1% entre los 32 y 39 años al 27,7%
entre los 60 y 70, casi siete veces más. Con la presión sistólica ocurre lo mismo: del 8,5% bajo 120 mmHg al
30,8% desde 160 mmHg. Ambas relaciones son monótonas y mucho más visibles que lo que sugieren sus correlaciones
lineales (r = 0,23 y 0,22), porque el riesgo de base es bajo y la correlación resume mal relaciones con un
desenlace binario poco frecuente.

## 3. Estimación de parámetros
Se estiman las medias de edad, colesterol total y presión sistólica con IC del 95%:
$\bar{x} \pm t_{0,975;n-1}s/\sqrt{n}$. Se usa t de Student porque la desviación poblacional es desconocida;
el tamaño muestral grande respalda la aproximación para medias aun con asimetría moderada.
La proporción de CHD usa Wilson como método principal y Wald como comparación.
Se utilizan casos disponibles. La inferencia supone observaciones independientes y una población de
referencia comparable a la cohorte; no garantiza representatividad de la población chilena.

#### Tabla 4 - Estimación puntual e intervalos de confianza al 95%

In [ ]:
z = stats.norm.ppf(0.975)

def ic_media(serie):
    x = serie.dropna()
    m, ee = x.mean(), x.std(ddof=1) / np.sqrt(len(x))
    critico = stats.t.ppf(0.975, df=len(x) - 1)
    return m, m - critico * ee, m + critico * ee, len(x)

def ic_proporcion(serie):
    x = serie.dropna()
    p, ni = x.mean(), len(x)
    ee = np.sqrt(p * (1 - p) / ni)
    return p, p - z * ee, p + z * ee, ni

filas = []
for nombre, col in [("Edad media (años)", "age"), ("Colesterol total medio (mg/dL)", "totChol"),
                    ("Presión sistólica media (mmHg)", "sysBP")]:
    m, lo, hi, ni = ic_media(df[col])
    filas.append({"Parámetro": nombre, "Estimación": round(m, 2), "IC 95% inf.": round(lo, 2),
                  "IC 95% sup.": round(hi, 2), "n": ni, "Mediana": df[col].median(), "Método": "t de Student"})
p, wald_lo, wald_hi, ni = ic_proporcion(df[TARGET])
w = stats.binomtest(int(df[TARGET].sum()), ni).proportion_ci(confidence_level=0.95, method="wilson")
lo, hi = w.low, w.high
filas.append({"Parámetro": "Proporción de CHD a 10 años (%)", "Estimación": round(p * 100, 2),
              "IC 95% inf.": round(lo * 100, 2), "IC 95% sup.": round(hi * 100, 2), "n": ni,
              "Mediana": np.nan, "Método": "Wilson"})
tabla_ic = pd.DataFrame(filas)
tabla_ic

In [ ]:
display(pd.DataFrame({"Método": ["Wilson (principal)", "Wald (comparación)"],
    "IC 95% inferior (%)": [w.low * 100, wald_lo * 100],
    "IC 95% superior (%)": [w.high * 100, wald_hi * 100]}).round(2))

**Lectura.** Los intervalos son estrechos por el tamaño de la muestra. Con un 95% de confianza, la edad media
poblacional está entre 49,33 y 49,84 años, el colesterol total medio entre 235,37 y 238,07 mg/dL, la presión
sistólica media entre 131,69 y 133,02 mmHg y la proporción de CHD a 10 años entre 14,15% y 16,31% (Wilson). Con este
tamaño de muestra el intervalo de Wilson prácticamente coincide con el de Wald.

**Limitaciones.** (1) El colesterol y la presión sistólica son asimétricos: el TCL respalda la normalidad de la
*media muestral*, pero la media queda algo por sobre el valor típico, por lo que se informa también la mediana
(234 mg/dL y 128 mmHg). (2) El colesterol se estima sin los 50 casos faltantes (1,2%); si la pérdida no fuera
aleatoria podría haber sesgo, su magnitud depende del mecanismo de ausencia, no solo de su porcentaje. (3) Los intervalos suponen que la muestra representa
a la población de interés; la cohorte de Framingham proviene de una sola ciudad de Estados Unidos, por lo que la
generalización a otras poblaciones debe hacerse con cautela.

## 4. Pruebas de hipótesis
Nivel de significancia $\alpha = 0{,}05$, umbral convencional fijado para esta fase exploratoria. Bajo $H_0$ y los supuestos de cada prueba,
controla el error de tipo I por contraste; no determina el error de tipo II ni la potencia. Se aplican tres pruebas según el tipo de variable y se añade un ajuste de Holm para esta familia
de contrastes. La prueba sobre faltantes es un diagnóstico separado, no una hipótesis sustantiva.

### 4.1 Prueba 1: edad entre quienes desarrollan CHD y quienes no (t de Welch)
$H_0:\ \mu_{\text{CHD}} = \mu_{\text{no CHD}}$ vs. $H_1:\ \mu_{\text{CHD}} \neq \mu_{\text{no CHD}}$.

In [ ]:
edad_chd = df.loc[df[TARGET] == 1, "age"]
edad_no = df.loc[df[TARGET] == 0, "age"]

# Verificación de supuestos
print("Verificación de supuestos")
for nombre, g in [("Con CHD", edad_chd), ("Sin CHD", edad_no)]:
    print(f"  {nombre}: n = {len(g):,}  media = {g.mean():.2f}  de = {g.std(ddof=1):.2f}  asimetría = {g.skew():.2f}")
lev = stats.levene(edad_chd, edad_no)
print(f"  Levene (igualdad de varianzas): W = {lev.statistic:.2f}, p = {lev.pvalue:.3f}")

res = stats.ttest_ind(edad_chd, edad_no, equal_var=False)
ic_dif = res.confidence_interval(confidence_level=0.95)
n1, n0 = len(edad_chd), len(edad_no)
sp = np.sqrt(((n1 - 1) * edad_chd.var(ddof=1) + (n0 - 1) * edad_no.var(ddof=1)) / (n1 + n0 - 2))
d = (edad_chd.mean() - edad_no.mean()) / sp
t_student = stats.ttest_ind(edad_chd, edad_no).statistic

print(f"\nt de Welch = {res.statistic:.2f}  (gl = {res.df:.0f})   valor p = {res.pvalue:.1e}")
print(f"Diferencia de medias = {edad_chd.mean() - edad_no.mean():.2f} años, IC 95% [{ic_dif.low:.2f}; {ic_dif.high:.2f}]")
print(f"d de Cohen = {d:.3f}   |   t de Student (referencia) = {t_student:.2f}")
print("Decisión:", "se rechaza H0" if res.pvalue < ALPHA else "no se rechaza H0")

**Lectura.** Se asume independencia entre personas; el archivo no permite verificar parentesco o agrupamiento, la edad es aproximadamente simétrica en
ambos (asimetría -0,25 y 0,32) y, con 644 y 3.594 personas, el TCL respalda la prueba *t*. Como los tamaños son
muy dispares, se usa la versión de Welch, que no asume varianzas iguales; Levene no detecta diferencia de
varianzas al 5% (p = 0,053), y la *t* de Student da la misma decisión (t = 15,05). La edad media es 54,15 años en
quienes desarrollan CHD y 48,77 en quienes no, una diferencia de 5,38 años (IC 95%: 4,70 a 6,06). El resultado
es significativo (t = 15,58; p < 0,001), por lo que **se rechaza $H_0$**, con un tamaño de efecto moderado
(d de Cohen = 0,64). En la práctica, la edad es un marcador de riesgo relevante, pero por sí sola no separa
nítidamente a ambos grupos.

### 4.2 Prueba 2: hipertensión prevalente y CHD ($\chi^2$ de independencia)
$H_0:$ hipertensión prevalente y CHD son independientes vs. $H_1:$ están asociadas.

In [ ]:
def chi2_2x2(var):
    """Prueba chi-cuadrado 2x2 con tamaños de efecto: V de Cramér, riesgo relativo y razón de momios."""
    tabla = pd.crosstab(df[var], df[TARGET])
    chi2, p, gl, esp = stats.chi2_contingency(tabla)          # con corrección de Yates (2x2)
    V = np.sqrt(stats.chi2_contingency(tabla, correction=False)[0] / tabla.values.sum())
    a, b = tabla.loc[1, 1], tabla.loc[1, 0]
    c, d_ = tabla.loc[0, 1], tabla.loc[0, 0]
    r1, r0 = a / (a + b), c / (c + d_)
    display(tabla)
    display(pd.DataFrame(esp, index=tabla.index, columns=tabla.columns).round(2))
    print(f"Frecuencia esperada mínima: {esp.min():.0f} ({'cumple' if esp.min() >= 5 else 'NO cumple'} el supuesto >= 5)")
    print(f"Chi2 = {chi2:.1f}   gl = {gl}   valor p = {p:.1e}   V de Cramér = {V:.3f}")
    print(f"% CHD con atributo = {r1 * 100:.1f}%   sin atributo = {r0 * 100:.1f}%")
    # IC log-normal del RR y del OR; las cuatro celdas son positivas en estos contrastes.
    se_rr = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d_))
    se_or = np.sqrt(1/a + 1/b + 1/c + 1/d_)
    rr, odds = r1/r0, (a*d_)/(b*c)
    rr_ic = np.exp(np.log(rr) + np.array([-1, 1])*z*se_rr)
    or_ic = np.exp(np.log(odds) + np.array([-1, 1])*z*se_or)
    dif = r1-r0
    se_dif = np.sqrt(r1*(1-r1)/(a+b) + r0*(1-r0)/(c+d_))
    display(pd.DataFrame({"Efecto": ["RR crudo", "OR cruda", "Diferencia (puntos porcentuales)"],
        "Estimación": [rr, odds, dif*100],
        "IC 95% inferior": [rr_ic[0], or_ic[0], (dif-z*se_dif)*100],
        "IC 95% superior": [rr_ic[1], or_ic[1], (dif+z*se_dif)*100]}).round(3))
    print("χ² con Yates; V de Cramér calculada con χ² sin corrección.")
    print(f"Riesgo relativo = {r1 / r0:.2f}   razón de momios = {(a * d_) / (b * c):.2f}")
    print("Decisión:", "se rechaza H0" if p < ALPHA else "no se rechaza H0")
    return {"chi2": chi2, "p": p, "V": V, "RR": r1 / r0, "OR": (a * d_) / (b * c), "r1": r1 * 100, "r0": r0 * 100}

res_hta = chi2_2x2("prevalentHyp")

**Lectura.** Se asume independencia entre personas y la frecuencia esperada mínima (200) supera ampliamente el
umbral de 5, así que el supuesto se cumple. La asociación es significativa ($\chi^2$ = 132,6; gl = 1;
p < 0,001), por lo que **se rechaza $H_0$**, con intensidad débil (V de Cramér = 0,18). El 24,7% de quienes
tienen hipertensión prevalente desarrolla CHD, frente al 10,9% de quienes no la tienen: un riesgo relativo de
2,26 y una razón de momios de 2,68. En la práctica, el riesgo observado es mayor en el grupo con hipertensión. La asociación es cruda,
no demuestra causalidad y debe evaluarse con ajuste por posibles factores de confusión.

### 4.3 Prueba 3: sexo y CHD ($\chi^2$ de independencia)
$H_0:$ sexo y CHD son independientes vs. $H_1:$ están asociados.

In [ ]:
res_sexo = chi2_2x2("male")

# Contexto: ¿los hombres difieren en edad, hipertensión o tabaquismo?
contexto = df.groupby("male")[["age", "prevalentHyp", "currentSmoker"]].mean()
contexto[["prevalentHyp", "currentSmoker"]] *= 100
contexto.index = ["Mujeres", "Hombres"]
contexto.columns = ["Edad media", "% hipertensión", "% fumador actual"]
contexto.round(1)

**Lectura.** La frecuencia esperada mínima (276) supera 5 y la asociación es significativa
($\chi^2$ = 32,6; gl = 1; p < 0,001), por lo que **se rechaza $H_0$**, con V de Cramér = 0,09.
El 18,9% de los hombres desarrolla CHD frente al 12,4% de las mujeres (RR crudo = 1,52; OR cruda = 1,64).
La edad media y la proporción de hipertensión son similares entre sexos, pero estas comparaciones
descriptivas no permiten descartar confusión. La asociación no está ajustada; no se demuestra una
señal independiente del sexo. La mayor frecuencia de tabaquismo masculino tampoco resuelve esta cuestión.

### 4.4 Comparación de decisiones con ajuste de Holm

In [ ]:
# Holm: control del error familiar para las tres hipótesis sustantivas
pvals = np.array([res.pvalue, res_hta["p"], res_sexo["p"]])
orden = np.argsort(pvals)
ajustados = np.empty(3)
ajustados[orden] = np.minimum(1, np.maximum.accumulate(pvals[orden]*np.arange(3,0,-1)))
display(pd.DataFrame({"Prueba": ["Edad (Welch)", "Hipertensión (χ²)", "Sexo (χ²)"],
    "p sin ajuste": pvals, "p Holm": ajustados, "Rechaza H0 con Holm": ajustados < ALPHA}))
print("Los tamaños de efecto son crudos y sus IC son individuales, no simultáneos.")

#### Figura 5 - Edad por grupo y tasa de CHD según hipertensión y sexo

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))

# (a) edad por grupo (rombo = media)
bp = ax[0].boxplot([edad_no, edad_chd], tick_labels=["Sin CHD", "Con CHD"], patch_artist=True, widths=0.5,
                   showmeans=True,
                   meanprops=dict(marker="D", markerfacecolor="white", markeredgecolor="#333333", markersize=6),
                   medianprops=dict(color="#333333", linewidth=1.4))
for patch, col in zip(bp["boxes"], [NAVY, RED]):
    patch.set_facecolor(col)
    patch.set_alpha(0.7)
ax[0].set_ylabel("Edad (años)")
ax[0].set_title("(a) Edad por grupo (rombo = media)")
ax[0].grid(axis="x", visible=False)

# (b) tasa de CHD según hipertensión y sexo
etiquetas = ["Sin\nhipertensión", "Con\nhipertensión", "Mujeres", "Hombres"]
tasas = [res_hta["r0"], res_hta["r1"], res_sexo["r0"], res_sexo["r1"]]
xx = np.array([0, 1, 2.6, 3.6])
ax[1].bar(xx, tasas, color=[NAVY, RED, NAVY, RED], width=0.6)
ax[1].axhline(gm, color=GREY, ls="--", lw=1.5, label=f"Media global = {fmt_es(gm)}%")
for xi, v in zip(xx, tasas):
    ax[1].text(xi, v + 0.5, f"{fmt_es(v)}%", ha="center", fontsize=9.5)
ax[1].set_xticks(xx)
ax[1].set_xticklabels(etiquetas)
ax[1].set_ylabel("% con CHD a 10 años")
ax[1].set_ylim(0, max(tasas) * 1.3)
ax[1].set_title("(b) Tasa de CHD según hipertensión y sexo")
ax[1].grid(axis="x", visible=False)
ax[1].legend(frameon=False, loc="upper left", fontsize=8.5)
eje_es(ax[1], "y")

plt.tight_layout()
plt.savefig(FIGURAS / "fig_hipotesis.png", bbox_inches="tight")
plt.show()

## 5. Síntesis y próximos pasos

- **Calidad:** sin duplicados ni inconsistencias; faltantes en siete variables (13,7% de las filas), concentrados
  en la glucosa (9,2%) sin evidencia de asociación con el objetivo, sin determinar el mecanismo de ausencia; valores extremos plausibles que se conservan.
- **Exploración:** objetivo desbalanceado (15,2% con CHD). El riesgo crece de forma monótona con la edad
  (4,1% a 27,7%) y con la presión sistólica (8,5% a 30,8%), aunque las correlaciones lineales con el objetivo
  son débiles (r ≤ 0,23). Hay predictores muy correlacionados entre sí (presión sistólica y diastólica, r = 0,78).
- **Estimación:** intervalos estrechos por el tamaño muestral; edad media 49,58 [49,33; 49,84] y proporción de
  CHD 15,20% [14,15%; 16,31%] (Wilson).
- **Pruebas:** la edad difiere entre grupos con efecto moderado (d = 0,64); se observan RR crudos de 2,26 para hipertensión
  y 1,52 para sexo masculino. Son asociaciones sin ajuste y no efectos causales.

**Próximos pasos (S2 y S3).** Imputar los faltantes (glucosa, educación y cigarrillos por día en fumadores),
evaluar el efecto de los valores extremos, elegir entre predictores colineales (presión sistólica, diastólica e
hipertensión prevalente), validar las estimaciones con remuestreo (*bootstrap*) y evaluar factores de confusión mediante análisis multivariable y entrenar modelos de
clasificación evaluados con métricas sensibles a la clase minoritaria.

In [ ]:
print("Figuras generadas:", sorted(f.name for f in FIGURAS.glob("fig_*.png")))